# 03 | Publish Gold marts
Only **Completed** orders count as realized sales. Returned and Cancelled orders remain in Silver for operational analysis but are excluded from the revenue metrics.

In [ ]:
from pyspark.sql import functions as F
for t in ["silver_orders", "silver_products", "silver_customers"]:
    assert spark.catalog.tableExists(t), f"Run 02 first: missing {t}"
orders = spark.table("silver_orders").filter(F.col("status") == "Completed")
products = spark.table("silver_products")
customers = spark.table("silver_customers")

daily = (orders.groupBy("order_date")
         .agg(F.countDistinct("order_id").alias("orders"),
              F.sum("quantity").alias("units_sold"),
              F.round(F.sum("line_amount"),2).alias("revenue"))
         .orderBy("order_date"))
product = (orders.join(products, "product_id")
           .groupBy("product_id", "product_name", "category")
           .agg(F.countDistinct("order_id").alias("orders"),
                F.sum("quantity").alias("units_sold"),
                F.round(F.sum("line_amount"),2).alias("revenue")))
customer = (orders.join(customers, "customer_id")
            .groupBy("customer_id", "customer_name", "region")
            .agg(F.countDistinct("order_id").alias("completed_orders"),
                 F.round(F.sum("line_amount"),2).alias("lifetime_revenue")))
for name,df in [("gold_daily_sales",daily), ("gold_product_performance",product),
                ("gold_customer_value",customer)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name,df.count())
